In [1]:
from google.colab import files

# Upload all required files
uploaded = files.upload()


Saving Screenshot 2025-11-24 173058.png to Screenshot 2025-11-24 173058.png
Saving uae_gb_non_oil_exports_model.joblib to uae_gb_non_oil_exports_model.joblib
Saving uae_gb_feature_names.joblib to uae_gb_feature_names.joblib
Saving clustering_results.csv to clustering_results.csv
Saving imbalance_analysis.csv to imbalance_analysis.csv


In [7]:
# Kill any process using port 8501
!fuser -k 8501/tcp


PREDICTIVE MODEL

In [9]:
# =========================
# STREAMLIT + NGROK DEPLOY
# =========================

import os, sys, subprocess, time, textwrap

# -------------------------------
# 1) Write the Streamlit app to app.py
# -------------------------------
app_code = textwrap.dedent(r'''
import numpy as np
import pandas as pd
import joblib
import streamlit as st
from datetime import datetime
import base64

@st.cache_resource
def load_model():
    model = joblib.load("uae_gb_non_oil_exports_model.joblib")
    feature_names = joblib.load("uae_gb_feature_names.joblib")
    return model, feature_names

gb_model, feature_names = load_model()

def build_features_from_history(export_history_kusd, feature_names):
    s = pd.Series(export_history_kusd, dtype=float)
    feat = {col: 0.0 for col in feature_names}
    feat["Export_kUSD"] = s.iloc[-1]
    feat["Export_lag1"] = s.iloc[-2] if len(s) >= 2 else 0.0
    feat["Export_lag2"] = s.iloc[-3] if len(s) >= 3 else 0.0
    prev = feat["Export_lag1"]
    feat["YoY"] = feat["Export_kUSD"] / prev - 1.0 if prev > 0 else 0.0
    return pd.DataFrame([feat])[feature_names]

def predict_next_year_export(history, last_year):
    X = build_features_from_history(history, feature_names)  # ✅ FIXED: pass feature_names
    pred_log = gb_model.predict(X)[0]
    pred_kusd = np.expm1(pred_log)
    current_kusd = history[-1]
    growth_rate = (pred_kusd - current_kusd) / current_kusd if current_kusd > 0 else 0.0
    next_year = last_year + 1
    return {
        "last_year": last_year,
        "next_year": next_year,
        "current_export_kUSD": current_kusd,
        "predicted_next_year_kUSD": pred_kusd,
        "predicted_growth_rate": growth_rate
    }

def growth_category(g):
    if g > 0.4:
        return "HIGH GROWTH 🚀"
    elif g > 0.1:
        return "MODERATE GROWTH 📈"
    elif g > 0:
        return "SLOW GROWTH ↗"
    else:
        return "STABLE / DECLINING ➡"

st.set_page_config(page_title="UAE Export Growth Predictor", page_icon="🇦🇪", layout="wide")

# -------------------------------
# Background + Custom Header Styling
# -------------------------------
def get_base64_image(image_path):
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode()

try:
    img_base64 = get_base64_image("Screenshot 2025-11-24 173058.png")

    st.markdown(f"""
    <style>
    html, body {{
        margin: 0;
        padding: 0;
        background-color: white;
    }}

    [data-testid="stAppViewContainer"] {{
        background-image: url("data:image/png;base64,{img_base64}");
        background-position: top center;
        background-repeat: no-repeat;
        background-size: cover;
        background-attachment: fixed;
        background-color: white;
    }}

    [data-testid="stHeader"], [data-testid="stToolbar"], header, .viewerBadge_container__1QSob {{
        display: none !important;
        visibility: hidden !important;
        height: 0 !important;
        position: absolute !important;
        top: -100px !important;
    }}

    .block-container {{
        padding-top: 0rem !important;
        margin-top: 0rem !important;
        background-color: rgba(255,255,255,0.85) !important;
        max-width: 1100px !important;
        border-radius: 12px;
    }}

    [data-testid="stMain"] {{
        background-color: transparent !important;
    }}

    [data-testid="stSidebar"] {{
        background-color: rgba(255,255,255,0.88) !important;
    }}
    </style>
    """, unsafe_allow_html=True)

    st.markdown("""
        <div style="
            width: 100%;
            background-color: rgba(255, 255, 255, 0.92);
            padding: 1rem 0;
            text-align: center;
            font-size: 1.85rem;
            font-weight: 600;
            color: #222;
            box-shadow: 0 2px 4px rgba(0,0,0,0.08);
            border-bottom: 1px solid rgba(0,0,0,0.06);
            margin-bottom: 2rem;
        ">
            🇦🇪 UAE Non-Oil Export Growth Predictor
        </div>
    """, unsafe_allow_html=True)

except Exception:
    st.warning("Background image could not be loaded.")

# -------------------------------
# Main Content Container
# -------------------------------
st.markdown("""
    <div style="padding: 2rem;">
""", unsafe_allow_html=True)

st.markdown("""
This tool forecasts **next-year non-oil exports** for a given HS-2 chapter using a trained Gradient Boosting model.
Enter your historical export data and instantly see the prediction for the upcoming year.
""")

# -------------------------------
# Inputs
# -------------------------------
c_inputs = st.columns([1,1,1])
with c_inputs[0]:
    hs_code = st.text_input("HS-2 Chapter Code (1–95)", value="25")
with c_inputs[1]:
    num_years = st.number_input("Number of years of history", min_value=3, max_value=12, value=4, step=1)
with c_inputs[2]:
    last_year = st.number_input("Last year of your data", min_value=1900, max_value=2100, value=2023, step=1)

years_list = list(range(last_year - num_years + 1, last_year + 1))

st.markdown("### Export history (in 1000 USD)")
history = []
cols = st.columns(4)
for i, year in enumerate(years_list):
    with cols[i % 4]:
        val = st.number_input(f"{year}", min_value=0.0, value=0.0, step=1000.0, key=f"exp_{year}")
        history.append(val)

# -------------------------------
# Prediction
# -------------------------------
if st.button("Predict next-year export"):
    try:
        if all(v == 0 for v in history):
            st.error("Please enter at least one non-zero export value.")
        else:
            result = predict_next_year_export(history, last_year)
            growth = result["predicted_growth_rate"]

            c1, c2 = st.columns(2)
            with c1:
                st.metric(
                    label=f"Current export ({result['last_year']})",
                    value=f"{result['current_export_kUSD']:,.0f} kUSD"
                )
            with c2:
                st.metric(
                    label=f"Predicted export ({result['next_year']})",
                    value=f"{result['predicted_next_year_kUSD']:,.0f} kUSD",
                    delta=f"{growth*100:,.2f}%"
                )

            st.success(f"Growth Category: {growth_category(growth)}")

            df = pd.DataFrame([{
                "HS-2 Chapter": hs_code,
                "Last Year": result["last_year"],
                "Current Export (kUSD)": result["current_export_kUSD"],
                "Next Year": result["next_year"],
                "Predicted Export (kUSD)": result["predicted_next_year_kUSD"],
                "Growth Rate (%)": growth * 100,
                "Growth Category": growth_category(growth)
            }])
            st.download_button(
                label="📥 Download Prediction as CSV",
                data=df.to_csv(index=False).encode("utf-8"),
                file_name=f"uae_export_prediction_{hs_code}_{result['next_year']}.csv",
                mime="text/csv"
            )

    except Exception as e:
        st.error(f"Error during prediction: {e}")

# -------------------------------
# Footer
# -------------------------------
st.markdown("---")
st.markdown(f"**Created by Iram**")

st.markdown("</div>", unsafe_allow_html=True)
''')

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written successfully.")

# -------------------------------
# 2) Install required packages
# -------------------------------
print("\n📦 Installing required libraries (this may take a bit)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "streamlit", "pyngrok", "joblib", "scikit-learn", "xgboost", "pandas", "numpy", "Pillow"])
print("✅ Libraries installed.")

# -------------------------------
# 3) Configure ngrok and start Streamlit
# -------------------------------
from pyngrok import ngrok

NGROK_TOKEN = "35qWeS38lQPOeTZ4qXnXdJWFhtA_sDfT9ng2p7keDC6prCfQ"  # <-- Your token

if NGROK_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    print("\n❌ Please edit the cell and set NGROK_TOKEN to your real token, then run again.")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    print("\n🔐 Ngrok auth token set.")

    ngrok.kill()

    print("\n🚀 Starting Streamlit app on port 8501...")
    streamlit_proc = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    time.sleep(8)

    print("🌐 Creating ngrok tunnel...")
    public_url = ngrok.connect(8501, "http").public_url
    print(f"\n✅ Public URL for your app: {public_url}\n")

    print("\n--- Streamlit logs (first 20 lines) ---")
    for _ in range(20):
        line = streamlit_proc.stdout.readline()
        if not line:
            break
        print(line, end="")
    print("\n---------------------------------------")


✅ app.py written successfully.

📦 Installing required libraries (this may take a bit)...
✅ Libraries installed.

🔐 Ngrok auth token set.

🚀 Starting Streamlit app on port 8501...
🌐 Creating ngrok tunnel...

✅ Public URL for your app: https://unejective-vulturelike-maudie.ngrok-free.dev


--- Streamlit logs (first 20 lines) ---



  You can now view your Streamlit app in your browser.

  URL: http://0.0.0.0:8501



KeyboardInterrupt: 

working

In [2]:
# ============================================
# 1) Install required packages
# ============================================
!pip install streamlit pyngrok pandas matplotlib

# ============================================
# 2) Write the Streamlit app to app.py
# ============================================
app_code = """
import pandas as pd
import streamlit as st
import base64

# -------------------------------
# Styling + Background
# -------------------------------
def get_base64_image(image_path):
    try:
        with open(image_path, "rb") as img_file:
            return base64.b64encode(img_file.read()).decode()
    except:
        return None

img_base64 = get_base64_image("Screenshot 2025-11-24 173058.png")
if img_base64:
    st.markdown(f'''
    <style>
    [data-testid="stAppViewContainer"] {{
        background-image: url("data:image/png;base64,{img_base64}");
        background-size: cover;
        background-position: top center;
        background-repeat: no-repeat;
        background-attachment: fixed;
    }}
    [data-testid="stHeader"], [data-testid="stToolbar"] {{
        display: none;
    }}
    .block-container {{
        background-color: rgba(255,255,255,0.85);
        border-radius: 12px;
        padding: 2rem;
        max-width: 1000px;
        margin: auto;
    }}
    .badge {{
        display: inline-block;
        padding: 0.35rem 0.6rem;
        border-radius: 999px;
        font-weight: 600;
        font-size: 0.9rem;
        margin-right: 0.4rem;
    }}
    .card {{
        padding: 1rem;
        background-color: #fff;
        border-radius: 10px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.08);
        border: 1px solid rgba(0,0,0,0.06);
    }}
    </style>
    ''', unsafe_allow_html=True)

st.markdown(\"\"\"
    <div style="
        text-align: center;
        font-size: 1.85rem;
        font-weight: 600;
        padding: 1rem;
        background-color: rgba(255,255,255,0.92);
        box-shadow: 0 2px 4px rgba(0,0,0,0.08);
        border-bottom: 1px solid rgba(0,0,0,0.06);
        margin-bottom: 2rem;
    ">
        🇦🇪 UAE Trade Dynamics — HS Chapter Analysis
    </div>
\"\"\", unsafe_allow_html=True)

# -------------------------------
# Load & prepare data
# -------------------------------
imbalance_df = pd.read_csv("imbalance_analysis.csv")
cluster_df = pd.read_csv("clustering_results.csv")
df = pd.merge(imbalance_df, cluster_df, on=["ProductCode", "ProductDescription"], how="left")

# -------------------------------
# Sidebar selection
# -------------------------------
st.sidebar.title("Select HS Chapter")
search_text = st.sidebar.text_input("Search by description (optional)")
filtered_codes = df["ProductCode"].unique()

if search_text.strip():
    mask = df["ProductDescription"].str.contains(search_text.strip(), case=False, na=False)
    filtered_codes = df.loc[mask, "ProductCode"].unique()
    if len(filtered_codes) == 0:
        st.sidebar.info("No match found. Showing all HS codes.")

selected_code = st.sidebar.selectbox("HS Chapter Code", sorted(filtered_codes))

# -------------------------------
# Retrieve selected row
# -------------------------------
sel_rows = df[df["ProductCode"] == selected_code]
if sel_rows.empty:
    st.error("Selected HS Chapter not found in data.")
    st.stop()

row = sel_rows.iloc[0]

# -------------------------------
# Visual mappings
# -------------------------------
severity_colors = {"Critical": "#e74c3c", "High": "#e67e22", "Medium": "#f1c40f", "Low": "#2ecc71"}
dep_colors = {"Transit-heavy": "#3498db", "Structural Dependency": "#9b59b6", "Other": "#7f8c8d"}
cluster_emojis = {"Stable": "🧊", "Developing": "🚀", "Declining": "⬇"}

severity = str(row["Deficit_Severity"])
dependency = str(row["Dependency_Type"])
category = str(row["Category"])
cluster_label = str(row["Cluster_Label"])
product_desc = str(row["ProductDescription"])
deficit_years = int(row["Deficit_Years"])
mean_net_trade = float(row["Mean_Net_Trade"])
reexport_ratio = float(row["Avg_Reexport_Ratio"])

severity_color = severity_colors.get(severity, "#95a5a6")
dependency_color = dep_colors.get(dependency, "#95a5a6")
cluster_icon = cluster_emojis.get(cluster_label, "❓")

# -------------------------------
# Header card
# -------------------------------
st.markdown(f\"\"\"
<div class="card">
  <h3 style="margin-top:0;">HS Chapter {selected_code}</h3>
  <div style="font-size:1.05rem; color:#333; margin-bottom:0.5rem;">{product_desc}</div>
  <div style="margin-top:0.5rem;">
    <span class="badge" style="background-color:{severity_color}; color:#fff;">Severity: {severity}</span>
    <span class="badge" style="background-color:{dependency_color}; color:#fff;">Dependency: {dependency}</span>
    <span class="badge" style="background-color:#2d3436; color:#fff;">Category: {category}</span>
    <span class="badge" style="background-color:#0984e3; color:#fff;">Cluster: {cluster_icon} {cluster_label}</span>
  </div>
</div>
\"\"\", unsafe_allow_html=True)

# -------------------------------
# Key metrics
# -------------------------------
c1, c2, c3 = st.columns(3)
with c1: st.metric("Deficit years", f"{deficit_years}")
with c2: st.metric("Mean net trade (USD)", f"{mean_net_trade:,.0f}")
with c3: st.metric("Avg re-export ratio (%)", f"{reexport_ratio:,.2f}")

st.markdown("#### Re-export intensity")
st.progress(min(max(reexport_ratio / 100.0, 0.0), 1.0))

# -------------------------------
# Interpretation (clean version)
# -------------------------------
st.markdown(\"\"\"
<div class="card" style="margin-top:1rem;">
  <h4 style="margin-top:0;">Interpretation</h4>
  <ul style="line-height:1.6;">
    <li>Higher severity signals sustained net import pressure, indicating a persistent trade deficit.</li>
    <li>Indicates structural reliance, such as transit through UAE or limited domestic production capacity.</li>
    <li>A concise classification that aligns with the nature of UAE’s dependency and trade behavior.</li>
    <li>Reflects overall trade dynamics for this chapter, showing whether the product is stable, developing, or declining.</li>
  </ul>
</div>
\"\"\", unsafe_allow_html=True)

# -------------------------------
# Download button
# -------------------------------
summary_df = pd.DataFrame([{
    "ProductCode": selected_code,
    "ProductDescription": product_desc,
    "Dependency_Type": dependency,
    "Deficit_Severity": severity,
    "Category": category,
    "Cluster_Label": cluster_label,
    "Deficit_Years": deficit_years,
    "Mean_Net_Trade": mean_net_trade,
    "Avg_Reexport_Ratio": reexport_ratio
}])

st.download_button(
    label="📥 Download Chapter Analysis",
    data=summary_df.to_csv(index=False).encode("utf-8"),
    file_name=f"hs_chapter_{selected_code}_analysis.csv",
    mime="text/csv"
)

# -------------------------------
# Footer
# -------------------------------
st.markdown("---")
st.markdown("**Created by Iram**")
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written successfully.")

# ============================================
# 3) Configure ngrok and start Streamlit
# ============================================
from pyngrok import ngrok
import subprocess, time

NGROK_TOKEN = "35qWeS38lQPOeTZ4qXnXdJWFhtA_sDfT9ng2p7keDC6prCfQ"  # your token

if not NGROK_TOKEN or NGROK_TOKEN.strip() == "":
    print("❌ NGROK_TOKEN is missing. Please provide a valid token.")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    ngrok.kill()

    print("🚀 Starting Streamlit app on port 8501...")
    streamlit_proc = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    time.sleep(8)  # wait for server to start

    print("🌐 Creating ngrok tunnel...")
    public_url = ngrok.connect(8501, "http").public_url
    print(f"\n✅ Your Streamlit app is live at: {public_url}\n")

    # Optional: show first few log lines
    for _ in range(15):
        line = streamlit_proc.stdout.readline()
        if not line:
            break
        print(line.strip())


✅ app.py written successfully.
🚀 Starting Streamlit app on port 8501...
🌐 Creating ngrok tunnel...

✅ Your Streamlit app is live at: https://unejective-vulturelike-maudie.ngrok-free.dev




You can now view your Streamlit app in your browser.

URL: http://0.0.0.0:8501



KeyboardInterrupt: 

**SINGLE VERSION**

In [11]:
import numpy as np
import pandas as pd
import joblib
import streamlit as st
import base64

st.set_page_config(page_title="UAE Trade Dashboard", page_icon="🇦🇪", layout="wide")

# -------------------------------
# Shared Styling + Background
# -------------------------------
def get_base64_image(image_path):
    try:
        with open(image_path, "rb") as img_file:
            return base64.b64encode(img_file.read()).decode()
    except:
        return None

img_base64 = get_base64_image("Screenshot 2025-11-24 173058.png")
if img_base64:
    st.markdown(f"""
    <style>
    [data-testid="stAppViewContainer"] {{
        background-image: url("data:image/png;base64,{img_base64}");
        background-size: cover;
        background-position: top center;
        background-repeat: no-repeat;
        background-attachment: fixed;
    }}
    [data-testid="stHeader"], [data-testid="stToolbar"] {{
        display: none !important;
    }}
    .block-container {{
        background-color: rgba(255,255,255,0.85);
        border-radius: 12px;
        padding: 2rem;
        max-width: 1100px;
        margin: auto;
    }}
    .badge {{
        display: inline-block;
        padding: 0.35rem 0.6rem;
        border-radius: 999px;
        font-weight: 600;
        font-size: 0.9rem;
        margin-right: 0.4rem;
    }}
    .card {{
        padding: 1rem;
        background-color: #fff;
        border-radius: 10px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.08);
        border: 1px solid rgba(0,0,0,0.06);
    }}
    </style>
    """, unsafe_allow_html=True)

# -------------------------------
# Tabs
# -------------------------------
tab1, tab2 = st.tabs(["📊 Export Growth Predictor", "🔍 HS Chapter Analysis"])

# -------------------------------
# TAB 1: Export Growth Predictor
# -------------------------------
with tab1:
    @st.cache_resource
    def load_model():
        model = joblib.load("uae_gb_non_oil_exports_model.joblib")
        feature_names = joblib.load("uae_gb_feature_names.joblib")
        return model, feature_names

    gb_model, feature_names = load_model()

    def build_features_from_history(export_history_kusd, feature_names):
        s = pd.Series(export_history_kusd, dtype=float)
        feat = {col: 0.0 for col in feature_names}
        feat["Export_kUSD"] = s.iloc[-1]
        feat["Export_lag1"] = s.iloc[-2] if len(s) >= 2 else 0.0
        feat["Export_lag2"] = s.iloc[-3] if len(s) >= 3 else 0.0
        prev = feat["Export_lag1"]
        feat["YoY"] = feat["Export_kUSD"] / prev - 1.0 if prev > 0 else 0.0
        return pd.DataFrame([feat])[feature_names]

    def predict_next_year_export(history, last_year):
        X = build_features_from_history(history, feature_names)
        pred_log = gb_model.predict(X)[0]
        pred_kusd = np.expm1(pred_log)
        current_kusd = history[-1]
        growth_rate = (pred_kusd - current_kusd) / current_kusd if current_kusd > 0 else 0.0
        next_year = last_year + 1
        return {
            "last_year": last_year,
            "next_year": next_year,
            "current_export_kUSD": current_kusd,
            "predicted_next_year_kUSD": pred_kusd,
            "predicted_growth_rate": growth_rate
        }

    def growth_category(g):
        if g > 0.4: return "HIGH GROWTH 🚀"
        elif g > 0.1: return "MODERATE GROWTH 📈"
        elif g > 0: return "SLOW GROWTH ↗"
        else: return "STABLE / DECLINING ➡"

    st.markdown("### UAE Non-Oil Export Growth Predictor")
    hs_code = st.text_input("HS-2 Chapter Code (1–95)", value="25")
    num_years = st.number_input("Number of years of history", min_value=3, max_value=12, value=4)
    last_year = st.number_input("Last year of your data", min_value=1900, max_value=2100, value=2023)

    years_list = list(range(last_year - num_years + 1, last_year + 1))
    history = []
    cols = st.columns(4)
    for i, year in enumerate(years_list):
        with cols[i % 4]:
            val = st.number_input(f"{year}", min_value=0.0, value=0.0, step=1000.0, key=f"exp_{year}")
            history.append(val)

    if st.button("Predict next-year export"):
        if all(v == 0 for v in history):
            st.error("Please enter at least one non-zero export value.")
        else:
            result = predict_next_year_export(history, last_year)
            growth = result["predicted_growth_rate"]

            c1, c2 = st.columns(2)
            with c1:
                st.metric("Current export", f"{result['current_export_kUSD']:,.0f} kUSD")
            with c2:
                st.metric("Predicted export", f"{result['predicted_next_year_kUSD']:,.0f} kUSD", delta=f"{growth*100:,.2f}%")

            st.success(f"Growth Category: {growth_category(growth)}")

            df = pd.DataFrame([{
                "HS-2 Chapter": hs_code,
                "Last Year": result["last_year"],
                "Current Export (kUSD)": result["current_export_kUSD"],
                "Next Year": result["next_year"],
                "Predicted Export (kUSD)": result["predicted_next_year_kUSD"],
                "Growth Rate (%)": growth * 100,
                "Growth Category": growth_category(growth)
            }])
            st.download_button(
                "📥 Download Prediction",
                df.to_csv(index=False).encode("utf-8"),
                f"uae_export_prediction_{hs_code}_{result['next_year']}.csv",
                "text/csv"
            )

# -------------------------------
# TAB 2: HS Chapter Analysis
# -------------------------------
with tab2:
    st.markdown("### UAE Trade Dynamics — HS Chapter Analysis")

    imbalance_df = pd.read_csv("imbalance_analysis.csv")
    cluster_df = pd.read_csv("clustering_results.csv")
    df = pd.merge(imbalance_df, cluster_df, on=["ProductCode", "ProductDescription"], how="left")

    st.sidebar.title("Select HS Chapter")
    search_text = st.sidebar.text_input("Search by description (optional)")
    filtered_codes = df["ProductCode"].unique()

    if search_text.strip():
        mask = df["ProductDescription"].str.contains(search_text.strip(), case=False, na=False)
        filtered_codes = df.loc[mask, "ProductCode"].unique()
        if len(filtered_codes) == 0:
            st.sidebar.info("No match found. Showing all HS codes.")

    selected_code = st.sidebar.selectbox("HS Chapter Code", sorted(filtered_codes))
    sel_rows = df[df["ProductCode"] == selected_code]
    if sel_rows.empty:
        st.error("Selected HS Chapter not found in data.")
        st.stop()

    row = sel_rows.iloc[0]
    severity_colors = {"Critical": "#e74c3c", "High": "#e67e22", "Medium": "#f1c40f", "Low": "#2ecc71"}
    dep_colors = {"Transit-heavy": "#3498db", "Structural Dependency": "#9b59b6", "Other": "#7f8c8d"}
    cluster_emojis = {"Stable": "🧊", "Developing": "🚀", "Declining": "⬇"}

    severity = str(row["Deficit_Severity"])
    dependency = str(row["Dependency_Type"])
    category = str(row["Category"])
    cluster_label = str(row["Cluster_Label"])
    product_desc = str(row["ProductDescription"])
    deficit_years = int(row["Deficit_Years"])
    mean_net_trade = float(row["Mean_Net_Trade"])
    reexport_ratio = float(row["Avg_Reexport_Ratio"])

    severity_color = severity_colors.get(severity, "#95a5a6")
    dependency_color = dep_colors.get(dependency, "#95a5a6")
    cluster_icon = cluster_emojis.get(cluster_label, "❓")

    st.markdown(f"""
    <div class="card">
      <h3 style="margin-top:0;">HS Chapter {selected_code}</h3>
      <div style="font-size:1.05rem; color:#333; margin-bottom:0.5rem;">{product_desc}</div>
      <div style="margin-top:0.5rem;">
        <span class="badge" style="background-color:{severity_color}; color:#fff;">Severity: {severity}</span>
        <span class="badge" style="background-color:{dependency_color}; color:#fff;">Dependency: {dependency}</span>
        <span class="badge" style="background-color:#2d3436; color:#fff;">Category: {category}</span>
        <span class="badge" style="background-color:#0984e3; color:#fff;">Cluster: {cluster_icon} {cluster_label}</span>
      </div>
    </div>
    """, unsafe_allow_html=True)

    c1, c2, c3 = st.columns(3)
    with c1: st.metric("Deficit years", f"{deficit_years}")
    with c2: st.metric("Mean net trade (USD)", f"{mean_net_trade:,.0f}")
    with c3: st.metric("Avg re-export ratio (%)", f"{reexport_ratio:,.2f}")

    st.markdown("#### Re-export intensity")
    st.progress(min(max(reexport_ratio / 100.0, 0.0), 1.0))

    st.markdown("""
    <div class="card" style="margin-top:1rem;">
      <h4 style="margin-top:0;">Interpretation</h4>
      <ul style="line-height:1.6;">
        <li>Higher severity signals sustained net import pressure, indicating a persistent trade deficit.</li>
        <li>Indicates structural reliance, such as transit through UAE or limited domestic production capacity.</li>
        <li>Classification aligns with the nature of dependency and trade behavior.</li>
        <li>Cluster reflects whether the chapter is stable, developing, or declining.</li>
      </ul>
    </div>
    """, unsafe_allow_html=True)

    summary_df = pd.DataFrame([{
        "ProductCode": selected_code,
        "ProductDescription": product_desc,
        "Dependency_Type": dependency,
        "Deficit_Severity": severity,
        "Category": category,
        "Cluster_Label": cluster_label,
        "Deficit_Years": deficit_years,
        "Mean_Net_Trade": mean_net_trade,
        "Avg_Reexport_Ratio": reexport_ratio
    }])

    st.download_button(
        label="📥 Download Chapter Analysis",
        data=summary_df.to_csv(index=False).encode("utf-8"),
        file_name=f"hs_chapter_{selected_code}_analysis.csv",
        mime="text/csv"
    )

# -------------------------------
# Footer
# -------------------------------
st.markdown("---")
st.markdown("**Created by Iram**")
from pyngrok import ngrok
import subprocess, time

NGROK_TOKEN = "35qWeS38lQPOeTZ4qXnXdJWFhtA_sDfT9ng2p7keDC6prCfQ"
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()

print("🚀 Starting Streamlit app on port 8501...")
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"]
)

time.sleep(8)  # wait for server

print("🌐 Creating ngrok tunnel...")
public_url = ngrok.connect(8501, "http").public_url
print(f"\n✅ Your Streamlit app is live at: {public_url}\n")


2025-11-24 20:01:18.878 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-24 20:01:18.920 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-24 20:01:18.921 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-24 20:01:18.923 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-24 20:01:18.924 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-24 20:01:18.926 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-24 20:01:18.927 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-24 20:01:18.935 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

🚀 Starting Streamlit app on port 8501...
🌐 Creating ngrok tunnel...

✅ Your Streamlit app is live at: https://unejective-vulturelike-maudie.ngrok-free.dev

